In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


In [2]:
# Constants
RANDOMSTATE = 1082

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOMSTATE)
# Set torch random seed
torch.manual_seed(RANDOMSTATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOMSTATE)
    torch.cuda.manual_seed_all(RANDOMSTATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
from gps_ids import protocol, schema

EXP = protocol.real_heldout("2d")
feature_names = EXP.feature_names


In [5]:
n_classes = len(EXP.labels)
scaler = EXP.scaler
X_train = EXP.X_train.values
X_test = EXP.X_test.values
y_train = EXP.y_train
y_test = EXP.y_test


### Two Neural Networks Compete

- Generator: This network tries to create fake data that looks like your real data.
- Discriminator: This network tries to tell the difference between real data and fake data.

### The Training Loop

- In each round (epoch), the generator produces some fake data.
- The discriminator looks at a mix of real and fake data and tries to classify each as real or fake.
- Both networks get feedback: the generator learns how to make better fakes, and the discriminator learns how to get better at spotting them.
- Both networks update their parameters using this feedback.

### The Loss Values

- Discriminator Loss (Loss_D): How well the discriminator is doing. Lower means is better.
- Generator Loss (Loss_G): How well the generator is fooling the discriminator. Lower means it's making more convincing fakes.
- Ideally, neither network gets too good or too bad. If one "wins" completely, the other can't learn.

In [6]:
# Define GAN
class Generator(nn.Module):
    def __init__(self, noise_dim, label_dim, data_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, data_dim)
        )

    def forward(self, z, labels_onehot):
        x = torch.cat([z, labels_onehot], dim=1)
        return self.model(x)


class Discriminator(nn.Module):
    def __init__(self, label_dim, data_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels_onehot):
        x = torch.cat([x, labels_onehot], dim=1)
        return self.model(x)


In [7]:
# Hyperparameters
noise_dim = 100
label_dim = n_classes
data_dim = X_train.shape[1]
epochs = 1000
batch_size = 1000
lr = 0.0002
betas = (0.5, 0.999)

G = Generator(noise_dim, label_dim, data_dim).to(device)
D = Discriminator(label_dim, data_dim).to(device)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

In [8]:
# DataLoader
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [9]:
# Training Loop
for epoch in range(epochs):
    for real_data, real_labels in train_loader:
        real_data = real_data.to(device)
        real_labels = real_labels.to(device)
        batch_sz = real_data.size(0)
        onehot_real = torch.nn.functional.one_hot(real_labels, num_classes=label_dim).float()

        # Train Discriminator
        optimizer_D.zero_grad()
        labels_real = torch.ones((batch_sz, 1), device=device)
        output_real = D(real_data, onehot_real)
        loss_real = criterion(output_real, labels_real)

        # Fake data
        noise = torch.randn(batch_sz, noise_dim, device=device)
        fake_labels = torch.randint(0, label_dim, (batch_sz,), device=device)
        onehot_fake = torch.nn.functional.one_hot(fake_labels, num_classes=label_dim).float()
        fake_data = G(noise, onehot_fake)
        labels_fake = torch.zeros((batch_sz, 1), device=device)
        output_fake = D(fake_data.detach(), onehot_fake)
        loss_fake = criterion(output_fake, labels_fake)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Train Generator
        optimizer_G.zero_grad()
        output_fake = D(fake_data, onehot_fake)
        loss_G = criterion(output_fake, labels_real)
        loss_G.backward()
        optimizer_G.step()

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss_D: {loss_D.item():.4f} | Loss_G: {loss_G.item():.4f}")

Epoch 1/1000 | Loss_D: 1.1949 | Loss_G: 0.8010


Epoch 20/1000 | Loss_D: 0.8402 | Loss_G: 1.4955


Epoch 40/1000 | Loss_D: 0.6762 | Loss_G: 1.8622


Epoch 60/1000 | Loss_D: 0.5529 | Loss_G: 2.0816


Epoch 80/1000 | Loss_D: 0.5860 | Loss_G: 2.0927


Epoch 100/1000 | Loss_D: 0.6045 | Loss_G: 2.1957


... (41 outputs left out)

Epoch 940/1000 | Loss_D: 0.5182 | Loss_G: 2.6589


Epoch 960/1000 | Loss_D: 0.5286 | Loss_G: 2.5517


Epoch 980/1000 | Loss_D: 0.5002 | Loss_G: 2.6519


Epoch 1000/1000 | Loss_D: 0.5225 | Loss_G: 2.4889


In [10]:
G.eval()
num_synth = 10000
with torch.no_grad():
    synth_labels = np.random.choice(np.arange(label_dim), num_synth)
    synth_labels_tensor = torch.tensor(synth_labels, dtype=torch.long, device=device)
    synth_labels_onehot = torch.nn.functional.one_hot(synth_labels_tensor, num_classes=label_dim).float()
    z = torch.randn(num_synth, noise_dim, device=device)
    generated = G(z, synth_labels_onehot).cpu().numpy()
    generated = scaler.inverse_transform(generated)

synthetic_df = pd.DataFrame(generated, columns=feature_names)
out_path = schema.write_synthetic(synthetic_df, synth_labels, "2d", "syntheticSpoofing_gan_data.csv")
print(f"Synthetic data saved to {out_path}")


Synthetic data saved to /workspace/project/Image-Based-Intrusion-Detection-System-for-Global-Positioning-System-Spoofing-in-Unmanned-Aerial-Veh/Datasets/excel/2dSpoofingDataset/syntheticSpoofing_gan_data.csv
